# 🎓 Educational Tutorial: Multi-Temporal Change Detection & Deforestation Mapping

This notebook serves as a self-contained learning artifact detailing the concepts, mathematics, and implementation of **multi-temporal change detection** to automatically identify deforestation from satellite imagery.

---

## 1. Core Concepts

### A. Why Temporal Analysis is Needed
Single-frame classification only tells us what is on the ground *now*. To monitor ecological processes like deforestation, urbanization, or agricultural expansion, we must perform **temporal analysis**—comparing the same geographic area at multiple times.

### B. Pixel-wise vs Patch-wise Comparison
* **Pixel-wise comparison**: High-resolution pixel comparisons. Extremely sensitive to registration offsets, illumination differences, and shadows, leading to high noise rates.
* **Patch-wise comparison**: Aggregates classifications into grids. By comparing patches (e.g. 64x64 blocks), we filter out high-frequency noise and focus on regional changes, which is highly robust for land cover mapping.

### C. Forest → Non-Forest Transition Definition
We define **deforestation** as any change where a patch is classified as `Forest` at time $T_1$ and transitions to any non-forest class (e.g. `AnnualCrop`, `Pasture`, `Residential`, `Highway`) at time $T_2$.

### D. Transition Matrix (Change Matrix)
A transition matrix $M_{i,j}$ counts the number of patches that transitioned from class $i$ to class $j$. The diagonal elements $M_{i,i}$ represent stable regions, while off-diagonals represent change.

In [ ]:
import numpy as np
import pandas as pd
import json
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from collections import Counter

print("Libraries loaded.")

In [ ]:
class ChangeDetector:
    def __init__(self, confidence_threshold=0.5):
        self.confidence_threshold = confidence_threshold
        self.classes = [
            "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
            "Industrial", "Pasture", "PermanentCrop", "Residential",
            "River", "SeaLake"
        ]
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        
    def detect_changes(self, map_a, map_b):
        classes_a = map_a["classes"]
        classes_b = map_b["classes"]
        confs_a = map_a.get("confidences", [1.0] * len(classes_a))
        confs_b = map_b.get("confidences", [1.0] * len(classes_b))
        
        changes = []
        for c_a, c_b, conf_a, conf_b in zip(classes_a, classes_b, confs_a, confs_b):
            if c_a == c_b:
                changes.append((c_a, c_b, False))
            else:
                if conf_a < self.confidence_threshold or conf_b < self.confidence_threshold:
                    changes.append((c_a, c_a, False)) # Reject change
                else:
                    changes.append((c_a, c_b, True))
        return changes
        
    def compute_matrix(self, changes):
        matrix = np.zeros((10, 10), dtype=np.int32)
        for class_a, class_b, _ in changes:
            idx_a = self.class_to_idx[class_a]
            idx_b = self.class_to_idx[class_b]
            matrix[idx_a, idx_b] += 1
        return matrix

In [ ]:
class DeforestationDetector:
    def __init__(self, forest_class="Forest"):
        self.forest_class = forest_class
        
    def detect_deforestation(self, changes):
        mask = []
        for class_a, class_b, _ in changes:
            is_deforested = (class_a == self.forest_class) and (class_b != self.forest_class)
            mask.append(1 if is_deforested else 0)
        return mask

In [ ]:
def calculate_forest_statistics(changes, deforestation_mask, patch_size_m=64.0):
    patch_area_ha = (patch_size_m * patch_size_m) / 10000.0
    classes_a = [c[0] for c in changes]
    classes_b = [c[1] for c in changes]
    
    forest_a_count = classes_a.count("Forest")
    forest_b_count = classes_b.count("Forest")
    
    forest_area_a_ha = forest_a_count * patch_area_ha
    forest_area_b_ha = forest_b_count * patch_area_ha
    
    deforested_count = sum(deforestation_mask)
    forest_loss_ha = deforested_count * patch_area_ha
    
    gain_count = sum(1 for c_a, c_b, _ in changes if c_a != "Forest" and c_b == "Forest")
    forest_gain_ha = gain_count * patch_area_ha
    
    percentage_loss = (deforested_count / max(1, forest_a_count)) * 100.0
    changed_patches = sum(1 for _, _, is_changed in changes if is_changed)
    stable_patches = len(changes) - changed_patches
    
    return {
        "forest_area_before_ha": forest_area_a_ha,
        "forest_area_after_ha": forest_area_b_ha,
        "forest_loss_ha": forest_loss_ha,
        "forest_gain_ha": forest_gain_ha,
        "percentage_forest_loss": percentage_loss,
        "changed_patches": changed_patches,
        "stable_patches": stable_patches
    }

In [ ]:
# Mock classification grids (Year A and Year B)
map_a = {
    "classes": ["Forest", "Forest", "Forest", "SeaLake", "Forest", "Forest", "Residential", "Pasture"],
    "confidences": [0.9, 0.8, 0.95, 0.7, 0.4, 0.85, 0.9, 0.8]
}

map_b = {
    "classes": ["AnnualCrop", "Forest", "Highway", "SeaLake", "AnnualCrop", "Forest", "Residential", "Pasture"],
    "confidences": [0.85, 0.8, 0.9, 0.7, 0.8, 0.85, 0.9, 0.8]
}

detector = ChangeDetector(confidence_threshold=0.5)
changes = detector.detect_changes(map_a, map_b)

defor_detector = DeforestationDetector()
mask = defor_detector.detect_deforestation(changes)

stats = calculate_forest_statistics(changes, mask)
print("Computed Forest Statistics:")
print(json.dumps(stats, indent=2))

# Build transition matrix
matrix = detector.compute_matrix(changes)
df_matrix = pd.DataFrame(matrix, index=detector.classes, columns=detector.classes)
print("\nTransition Matrix:")
display(df_matrix)